# Hone SN5 ARC Miner LLM Lab

This notebook tests a local Hugging Face instruct model against Hone synthetic ARC tasks. Do not paste wallet keys, seed phrases, Bittensor private keys, or paid API keys into Colab.

In [ ]:
#@title Configuration
REPO_URL = "https://github.com/nudiltoys-cmyk/hone-arc-miner"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  #@param {type:"string"}
LOAD_4BIT = False  #@param {type:"boolean"}
N_PROBLEMS = 3  #@param {type:"integer"}
SEED = 7  #@param {type:"integer"}
CHAIN_MIN = 3  #@param {type:"integer"}
CHAIN_MAX = 7  #@param {type:"integer"}
MAX_NEW_TOKENS = 1800  #@param {type:"integer"}

For a GPU runtime, try `Qwen/Qwen2.5-7B-Instruct` with `LOAD_4BIT=True`. On CPU, keep the default 1.5B model and small `N_PROBLEMS`.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORKDIR = Path("/content")
for name in ("hone-arc-miner", "research-hone"):
    path = WORKDIR / name
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "hone-arc-miner"], cwd=WORKDIR, check=True)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/manifold-inc/hone", "research-hone"], cwd=WORKDIR, check=True)
os.chdir(WORKDIR / "hone-arc-miner")
print(Path.cwd())

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "solver/requirements.txt"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "transformers>=4.51.0", "accelerate>=1.6.0", "sentencepiece"], check=True)
if LOAD_4BIT:
    subprocess.run(["python", "-m", "pip", "install", "-q", "bitsandbytes>=0.43.0"], check=True)
subprocess.run(["python", "-m", "py_compile", "solver/arc_solver.py", "tools/evaluate_hf_llm.py"], check=True)
print("compile ok")

In [ ]:
import shutil

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No nvidia-smi found; running on CPU runtime")

In [ ]:
json_out = f"/content/hone_llm_eval_n{N_PROBLEMS}_seed{SEED}.json"
cmd = [
    "python", "tools/evaluate_hf_llm.py",
    "--n", str(N_PROBLEMS),
    "--seed", str(SEED),
    "--chain-min", str(CHAIN_MIN),
    "--chain-max", str(CHAIN_MAX),
    "--model-id", MODEL_ID,
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--json-out", json_out,
]
if LOAD_4BIT:
    cmd.append("--load-4bit")
subprocess.run(cmd, check=True)
print("report:", json_out)

In [ ]:
import json
from pathlib import Path

report = json.loads(Path(json_out).read_text())
summary = {k: report[k] for k in [
    "n", "seed", "chain_min", "chain_max", "model_id", "load_4bit",
    "exact_matches", "exact_match_rate", "shape_match_rate",
    "avg_partial_correctness", "avg_grid_similarity", "elapsed_s",
]}
summary